In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

# 1. LOAD DATA
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

# 2. FEATURE ENGINEERING
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Speed_Score'] = df['Weight'] / df['Sprint_40yd']
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Power_Index'] = df['Vertical_Jump'] * df['Weight']
    df['Total_Agility'] = df['Agility_3cone'] + df['Shuttle']
    df['Agility_Ratio'] = df['Agility_3cone'] / df['Shuttle']
    df['Strength_Index'] = df['Bench_Press_Reps'] / df['Weight']
    df['Speed_pos_type_diff'] = df['Sprint_40yd'] - df.groupby('Position_Type')['Sprint_40yd'].transform('mean')
    for col in ['Agility_3cone', 'Shuttle', 'Bench_Press_Reps']:
        df[f'{col}_missing'] = df[col].isnull().astype(int)
    df['Catch_Radius'] = df['Height'] + (df['Vertical_Jump']/100) + (df['Broad_Jump']/100) - (df['Shuttle']/10)
    df['Height_Adj_Speed'] = (df['Weight'] * (df['Height'] ** 0.5)) / (df['Sprint_40yd'] ** 4)
    for stat in ['Sprint_40yd', 'BMI', 'Catch_Radius']:
        df[f'{stat}_pos_diff'] = df[stat] - df.groupby('Position')[stat].transform('mean')
    df['Jump_Rank'] = df.groupby('Position')['Vertical_Jump'].rank(ascending=False)
    df['Speed_Rank'] = df.groupby('Position')['Sprint_40yd'].rank(ascending=True)
    df['Combined_Rank'] = df['Jump_Rank'] + df['Speed_Rank']
    df['Size_Agility'] = df['BMI'] * df['Total_Agility']

# 3. PREPROCESSING
school_freq = train['School'].value_counts().to_dict()
train['School_Freq'] = train['School'].map(school_freq)
test['School_Freq'] = test['School'].map(school_freq).fillna(0)

train = train.drop(columns=["Id", "School"])
test = test.drop(columns=["Id", "School"])

le = LabelEncoder()
for col in ["Player_Type", "Position_Type", "Position"]:
    train[col] = le.fit_transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

X = train.drop(columns=["Drafted"])
y = train["Drafted"]

from category_encoders.target_encoder import TargetEncoder
from sklearn.model_selection import StratifiedKFold

# Define categorical columns
cat_cols = ["Player_Type", "Position_Type", "Position"]

# Target encoding with CV
te = TargetEncoder(cols=cat_cols, smoothing=0.3)

# Fit-transform on train with CV to avoid leakage
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2025)
X_te = X.copy()
for train_idx, valid_idx in skf.split(X, y):
    X_te = X.copy().astype(float)

    for train_idx, valid_idx in skf.split(X, y):
        te.fit(X.iloc[train_idx, :], y.iloc[train_idx])
        X_te.iloc[valid_idx, :] = te.transform(X.iloc[valid_idx, :])

    test_te = te.transform(test).astype(float)

# Transform test using encoder fitted on full train
te.fit(X, y)
test_te = te.transform(test)

# Replace imputation step with encoded data
imputer = IterativeImputer(random_state=2025)
X_imputed = pd.DataFrame(imputer.fit_transform(X_te), columns=X_te.columns)
test_imputed = pd.DataFrame(imputer.transform(test_te), columns=test_te.columns)


# 4. MODELS
cat_model = CatBoostClassifier(
    iterations=250, depth=7, learning_rate=0.05,
    subsample=0.9, bootstrap_type='Bernoulli',
    random_seed=2025, verbose=0
)
xgb_model = XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.05,
    random_state=2025, eval_metric='logloss'
)
lgbm_model = LGBMClassifier(
    n_estimators=200, learning_rate=0.05,
    random_state=2025, objective='binary'
)

# 5. CV
rskf = RepeatedStratifiedKFold(
    n_splits=20, n_repeats=3, random_state=42
)

# 6. OOF ARRAYS
oof_cat  = np.zeros(len(X_imputed))
oof_xgb  = np.zeros(len(X_imputed))
oof_lgbm = np.zeros(len(X_imputed))
test_cat  = np.zeros(len(test_imputed))
test_xgb  = np.zeros(len(test_imputed))
test_lgbm = np.zeros(len(test_imputed))

# 7. FOLD LOOP
for fold, (train_idx, valid_idx) in enumerate(
    rskf.split(X_imputed, y)):

    X_t = X_imputed.iloc[train_idx]
    X_v = X_imputed.iloc[valid_idx]
    y_t = y.iloc[train_idx]

    cat_model.fit(X_t, y_t)
    xgb_model.fit(X_t, y_t)
    lgbm_model.fit(X_t, y_t)

    oof_cat[valid_idx]  = cat_model.predict_proba(X_v)[:, 1]
    oof_xgb[valid_idx]  = xgb_model.predict_proba(X_v)[:, 1]
    oof_lgbm[valid_idx] = lgbm_model.predict_proba(X_v)[:, 1]

    test_cat  += cat_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    test_xgb  += xgb_model.predict_proba(test_imputed)[:, 1] / (20 * 3)
    test_lgbm += lgbm_model.predict_proba(test_imputed)[:, 1] / (20 * 3)

# 8. RANK AVERAGING
rank_cat  = rankdata(test_cat)  / len(test_cat)
rank_xgb  = rankdata(test_xgb)  / len(test_xgb)
rank_lgbm = rankdata(test_lgbm) / len(test_lgbm)

final_preds = (rank_cat + rank_xgb + rank_lgbm) / 3

# 9. SUBMISSION
submission = sample_sub.copy()
submission["Drafted"] = final_preds
submission.to_csv("submission_rank_avg.csv", index=False)
print("Done!")

TypeError: Invalid value '[0.71228771 0.71228771 0.6209607  0.6209607  0.6209607  0.6209607
 0.6209607  0.6209607  0.71228771 0.6209607  0.71228771 0.71228771
 0.71228771 0.71228771 0.6209607  0.6209607  0.71228771 0.71228771
 0.71228771 0.6209607  0.6209607  0.6209607  0.6209607  0.71228771
 0.6209607  0.71228771 0.71228771 0.6209607  0.6209607  0.6209607
 0.6209607  0.71228771 0.71228771 0.6209607  0.6209607  0.71228771
 0.71228771 0.71228771 0.71228771 0.71228771 0.6209607  0.6209607
 0.71228771 0.6209607  0.6209607  0.6209607  0.6209607  0.6209607
 0.6209607  0.71228771 0.71228771 0.71228771 0.6209607  0.6209607
 0.23076923 0.6209607  0.71228771 0.71228771 0.6209607  0.6209607
 0.71228771 0.6209607  0.6209607  0.71228771 0.71228771 0.6209607
 0.6209607  0.23076923 0.6209607  0.6209607  0.71228771 0.6209607
 0.6209607  0.6209607  0.71228771 0.71228771 0.6209607  0.71228771
 0.6209607  0.6209607  0.23076923 0.6209607  0.6209607  0.6209607
 0.6209607  0.6209607  0.71228771 0.71228771 